# MLOps Training 2026/2027 — Task 2

## Notebook 5 — Feature Engineering

This notebook builds leakage-safe features. It fits all imputers, encoders, and scalers on the training split only, then applies the fitted transformer to validation and test.

**Inputs:** `03_train.parquet`, `03_validation.parquet`, `03_test.parquet`

**Outputs:** sparse feature matrices, fitted transformers, and a feature list.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\HP\Desktop\MLOps")
TASK2_ROOT = PROJECT_ROOT / "Task2"
ARTIFACTS_DIR = TASK2_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

import joblib
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_parquet(ARTIFACTS_DIR / "tables" / "03_train.parquet")
validation = pd.read_parquet(ARTIFACTS_DIR / "tables" / "03_validation.parquet")
test = pd.read_parquet(ARTIFACTS_DIR / "tables" / "03_test.parquet")


## 1. Define prediction-time features

The target and post-outcome columns are removed. Date columns are transformed into components and pre-delivery durations. Identifiers are excluded to avoid memorization and high-cardinality noise.


In [2]:
def make_features(frame):
    x = frame.copy()
    date_cols = ["order_purchase_timestamp", "order_approved_at", "order_estimated_delivery_date"]
    for col in date_cols:
        x[col] = pd.to_datetime(x[col], errors="coerce")
    x["purchase_year"] = x["order_purchase_timestamp"].dt.year
    x["purchase_month"] = x["order_purchase_timestamp"].dt.month
    x["purchase_dayofweek"] = x["order_purchase_timestamp"].dt.dayofweek
    x["purchase_hour"] = x["order_purchase_timestamp"].dt.hour
    x["approval_hours"] = (x["order_approved_at"] - x["order_purchase_timestamp"]).dt.total_seconds() / 3600
    x["estimated_window_days"] = (x["order_estimated_delivery_date"] - x["order_purchase_timestamp"]).dt.total_seconds() / 86400
    drop_cols = [
        "is_late", "order_id", "customer_id", "customer_unique_id",
        "order_status", "order_delivered_carrier_date",
        "order_delivered_customer_date", "avg_review_score",
        "review_record_count", "order_purchase_timestamp",
        "order_approved_at", "order_estimated_delivery_date"
    ]
    return x.drop(columns=[c for c in drop_cols if c in x.columns])

X_train_raw = make_features(train)
X_validation_raw = make_features(validation)
X_test_raw = make_features(test)
y_train = train["is_late"].astype("int8").to_numpy()
y_validation = validation["is_late"].astype("int8").to_numpy()
y_test = test["is_late"].astype("int8").to_numpy()

numeric_features = X_train_raw.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train_raw.columns if c not in numeric_features]
print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")


Numeric features: 19
Categorical features: 2


## 2. Fit preprocessing on train only


In [3]:
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numeric_features), ("categorical", categorical_pipeline, categorical_features)], remainder="drop")
X_train = preprocessor.fit_transform(X_train_raw)
X_validation = preprocessor.transform(X_validation_raw)
X_test = preprocessor.transform(X_test_raw)
print("Transformed shapes:", X_train.shape, X_validation.shape, X_test.shape)


Transformed shapes: (67529, 802) (14470, 802) (14471, 802)


## 3. Save feature artifacts


In [4]:
sparse.save_npz(ARTIFACTS_DIR / "features" / "05_X_train.npz", X_train.tocsr() if sparse.issparse(X_train) else sparse.csr_matrix(X_train))
sparse.save_npz(ARTIFACTS_DIR / "features" / "05_X_validation.npz", X_validation.tocsr() if sparse.issparse(X_validation) else sparse.csr_matrix(X_validation))
sparse.save_npz(ARTIFACTS_DIR / "features" / "05_X_test.npz", X_test.tocsr() if sparse.issparse(X_test) else sparse.csr_matrix(X_test))
np.save(ARTIFACTS_DIR / "features" / "05_y_train.npy", y_train)
np.save(ARTIFACTS_DIR / "features" / "05_y_validation.npy", y_validation)
np.save(ARTIFACTS_DIR / "features" / "05_y_test.npy", y_test)
joblib.dump(preprocessor, ARTIFACTS_DIR / "features" / "05_transformers.joblib")
feature_list = list(preprocessor.get_feature_names_out())
(ARTIFACTS_DIR / "features" / "05_feature_list.json").write_text(json.dumps(feature_list, indent=2), encoding="utf-8")
metadata = {"feature_count": len(feature_list), "numeric_features": numeric_features, "categorical_features": categorical_features, "excluded_future_columns": ["order_delivered_customer_date", "avg_review_score", "review_record_count"]}
(ARTIFACTS_DIR / "reports" / "05_feature_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved {len(feature_list)} transformed features and fitted preprocessing objects.")


Saved 802 transformed features and fitted preprocessing objects.


In [5]:
assert len(feature_list) > 0
assert X_train.shape[0] == len(y_train)
assert X_validation.shape[0] == len(y_validation)
assert X_test.shape[0] == len(y_test)
assert (ARTIFACTS_DIR / "features" / "05_transformers.joblib").exists()
print("ALL NOTEBOOK 5 CHECKS PASSED")


ALL NOTEBOOK 5 CHECKS PASSED
